In [89]:
import pathlib
import re
# import argparse

# import yaml
# import pandas as pd
import numpy as np
import scipy
from sklearn.model_selection import train_test_split
import pandas as pd

import jax
import jax.numpy as jnp
import jax.random as jr
import optax
import equinox as eqx
from jaxtyping import Array, Float, Int, PyTree

# from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader, Subset

import altair as alt

import trapallada.data

In [2]:
# input_directory = pathlib.Path.cwd().parent / "light curves" / "Simulations"
input_directory = pathlib.Path.cwd().parent / "light curves"
assert input_directory.exists()

In [3]:
base_filename = 'small'

signals_file = input_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

metadata_file = input_directory / f'{base_filename}_labels.parquet'
assert metadata_file.exists

In [4]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
len(ds)

18000

In [5]:
# re_label = re.compile(r'.*_s(\d+)_(\d+)\.mat')
# re_label.match('AM_s1_284.mat').groups()

In [6]:
i = ds.metadata_df[ds.metadata_df['label'] == 'AM'].index[0]
ds.metadata_df.loc[i]

label                    AM
std                     2.0
file name    AM/AM_s2_3.jpg
Name: 12000, dtype: object

In [7]:
# am_files = (input_directory / 'AM').rglob('*.mat')
# y = scipy.io.loadmat(next(iter(am_files)))['y']
# y = scipy.io.loadmat(input_directory / 'Simulations' / 'AM' /  'AM_s2_3.mat')['y']
y = ds[i][1]
df = pd.DataFrame(y.squeeze(), columns=['signal'])
df


,signal
0,2.167470
1,2.730022
2,2.250210
3,2.036631
4,2.202767
...,...
495,2.698509
496,3.435666
497,4.214709
498,2.854706


In [8]:
alt.Chart(df.reset_index()).mark_line().encode(
    x='index', y='signal'
)

alt.Chart(...)

In [9]:
# def read_directory(dir: str | pathlib.Path):
    
#     ys = []
#     cols = []
    
#     # for every file in the directory
#     for f in dir.rglob('*.mat'):
        
#         ys.append(scipy.io.loadmat(f)['y'])

#         sigma, id = re_label.match(f.name).groups()

#         cols.append(f's{sigma}_{id}')
    
#     return pd.DataFrame(np.vstack(ys).T, columns=cols)

In [10]:
# df = read_directory(input_directory / 'AM')
# df.head

In [11]:
# ds[:3][1].squeeze().shape

In [165]:
rng_key = jr.PRNGKey(42)

parameters = {
    'training': {
        'splitting': {
            'test_size':  0.2,
            'random_state': 42
        },
        'batch_size': 32,
        'learning_rate': 1e-3,
    },
}

In [13]:
train_idx, validation_idx = train_test_split(
    np.arange(len(ds)), 
    **parameters['training']['splitting'], 
    shuffle=True,
    stratify=ds.labels
)
train_ds, validation_ds = Subset(ds, train_idx), Subset(ds, validation_idx)

# a random key...
key, subkey = jr.split(rng_key)

# ...to compute a seed for a PyTorch `Generator`...
torch_seed = jr.bits(subkey, (1,), dtype=jnp.uint32).item()
torch_rng = torch.Generator().manual_seed(torch_seed)

# ...meant for "batching" in the `DataLoader`'s
train_dl = DataLoader(train_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)
validation_dl = DataLoader(validation_ds, batch_size=parameters['training']['batch_size'], shuffle=False, generator=torch_rng)

In [14]:
ds.metadata_df.head()

,label,std,file name
0,Constant,1.0,Constant/cLC_s1_793.jpg
1,Constant,3.0,Constant/cLC_s3_524.jpg
2,Constant,3.0,Constant/cLC_s3_956.jpg
3,Constant,3.0,Constant/cLC_s3_283.jpg
4,Constant,5.0,Constant/cLC_s5_666.jpg


The indexes (1st element in the `tuple` for a batch) are used to build a `DataFrame`

In [65]:
batch = next(iter(train_dl))
batch_metadata_df = ds.metadata_df.loc[batch[0]]
# df = pd.DataFrame(batch[1].squeeze().T, columns=batch_metadata_df['label'].astype(str) + '_' + batch_metadata_df['std'].astype(int).astype(str))
df = pd.DataFrame(batch[1].squeeze().T, columns=batch_metadata_df['file name'].str.extract(r'.*/(.*).jpg')[0])
df

,AM_s2_906,FM_s5_307,FM_s4_740,AM_s5_238,cLC_s5_586,cLC_s5_248,FM_s1_465,cLC_s10_135,AM_s5_248,cLC_s3_507,...,cLC_s3_312,cLC_s10_232,AM_s4_999,cLC_s10_645,FM_s1_634,cLC_s1_624,FM_s10_295,FM_s2_497,AM_s4_499,cLC_s3_488
0,9.974388,6.440574,21.569034,41.647796,0.682139,-10.402873,48.295051,-4.344237,12.962303,-0.719106,...,2.396088,5.263309,26.227355,14.428149,32.247807,-0.266728,-0.333451,45.865302,27.144221,1.804945
1,7.854935,2.286819,15.458958,26.643542,1.765054,3.449168,36.394702,1.364026,1.409253,2.674147,...,3.301413,2.958244,19.423850,2.024228,-26.569551,-1.139272,4.516658,42.684193,15.984727,2.154495
2,7.401620,-7.160564,2.847059,26.225561,7.610417,10.902963,5.701338,0.137194,-7.231247,1.284064,...,-0.662248,8.021483,13.711669,3.391752,3.818997,0.336572,6.947814,37.804741,-21.525929,4.619043
3,3.232806,-10.552812,-12.458677,13.875726,-3.478585,6.956614,-28.808371,6.937356,-7.569361,2.933081,...,-2.810892,-13.968496,-3.436231,11.718859,26.971862,2.015155,-1.964616,30.685803,-19.964228,3.498551
4,8.362470,-3.610619,-19.208595,10.778331,8.769910,4.765773,-47.111876,-4.254733,-8.804109,10.556660,...,-3.462929,15.194070,-16.842417,-22.602688,-21.434424,2.133179,-7.194952,23.829883,-11.151784,5.641507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1.733834,-5.377999,14.896126,-9.299699,11.358340,13.129894,-47.707096,43.277120,-6.016494,2.383902,...,19.312984,-19.193088,-4.117230,10.271917,-20.711352,-1.107174,-6.563976,-45.340312,21.346799,-30.218504
496,2.435327,5.475090,21.096056,-17.252791,18.327328,12.996798,-46.278853,38.013375,11.763200,-0.998850,...,15.808646,-43.097418,2.689663,-38.174628,-21.384850,-0.643997,6.519577,-44.854401,1.811497,-30.090864
497,1.508797,3.800788,11.286786,-23.122552,13.593749,11.909723,-43.464088,33.140822,-19.459222,3.659185,...,16.339490,-30.585516,-0.418671,-7.843843,20.279172,0.287928,13.572465,-43.422261,-17.726189,-30.450324
498,2.683124,5.214490,-1.875918,-30.716621,17.815532,-4.181668,-43.504098,62.137035,-7.788140,6.168490,...,14.030261,-10.636155,3.328387,-13.287250,33.089044,-0.394951,7.401115,-44.693165,-14.188805,-30.210021


In [68]:
# df = df.iloc[:, :2]
alt.Chart(
    df.reset_index()
).transform_fold(
    df.columns.to_list()
).mark_line().encode(
    x='index:Q', y='value:Q', color='key:N'
).properties(width=1_200, height=400)

alt.Chart(...)

In [69]:
# df_sample = df.sample(n=50, axis=1)
# alt.Chart(
#     df_sample.reset_index()
# ).transform_fold(
#     df_sample.columns.to_list()
# ).mark_line().encode(
#     x='index:Q', y='value:Q', color='key:N'
# ).properties(width=1_200, height=400)

In [195]:
x = df.iloc[:, 0:1].values.T
in_channels = 1
x.shape

(1, 500)

In [196]:
stride = 4

In [197]:
out_channels = 2
conv = eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=60, stride=stride, key=jr.PRNGKey(42))
in_channels = out_channels
pool = eqx.nn.MaxPool1d(kernel_size=2)
# x = conv(x)
x = jax.nn.relu(pool(conv(x)))
x.shape

(2, 110)

In [199]:
out_channels = 4
conv = eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=25, stride=stride, key=jr.PRNGKey(42))
in_channels = out_channels
x = jax.nn.relu(pool(conv(x)))
x.shape

(4, 21)

In [200]:
out_channels = 8
conv = eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=10, stride=stride, key=jr.PRNGKey(42))
in_channels = out_channels
x = jax.nn.relu(pool(conv(x)))
x.shape

(8, 2)

In [202]:
x = jnp.ravel(x)
x.shape

In [208]:
linear = eqx.nn.Linear(in_features=np.prod(x.shape), out_features=3, key=jr.PRNGKey(42))
linear(x)

Array([-0.99535525, -0.08502249,  0.19692653], dtype=float32)

In [130]:
jr.split(jr.PRNGKey(42))

Array([[2465931498, 3679230171],
       [ 255383827,  267815257]], dtype=uint32)

In [131]:
class Classifier(eqx.Module):

    layers: list

    def __init__(self, rng_key):

        # key1, key2, key3, key4, key5, key6, key7, key8 = jr.split(rng_key, 8)

        stride = 4

        # self.layers = [
        #     eqx.nn.Conv1d(in_channels=1, out_channels=2, kernel_size=60, stride=stride, key=key1),
        #     eqx.nn.MaxPool1d(kernel_size=2),
        #     jax.nn.relu,
        #     #
        #     eqx.nn.Conv1d(in_channels=2, out_channels=4, kernel_size=25, stride=stride, key=key2),
        #     eqx.nn.MaxPool1d(kernel_size=2),
        #     jax.nn.relu,
        #     #
        #     eqx.nn.Conv1d(in_channels=4, out_channels=8, kernel_size=10, stride=stride, key=key3),
        #     eqx.nn.MaxPool1d(kernel_size=2),
        #     jax.nn.relu,
        # ]

        self.layers = []

        channels = [1, 2, 4, 8]
        kernel_sizes = [50, 30, 15]

        for in_channels, out_channels, key, kernel_size in zip(
            channels[:-1], channels[1:],
            jr.split(jr.PRNGKey(42), len(channels) - 1),
            kernel_sizes
        ):
            
            self.layers.extend([
                eqx.nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, key=key),
                eqx.nn.MaxPool1d(kernel_size=2),
                jax.nn.relu,
            ])

        self.layers.append(jnp.squeeze)

        key, subkey = jr.split(key)
        self.layers.append(eqx.nn.Linear(in_features=channels[-1], out_features=3, key=subkey))
        
    def __call__(self, x: Float[Array, '1 length'] ) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)

        return x

    def debug_call(self, x: Float[Array, '1 length'] ) -> Float[Array, 'twice_dim_z']:

        for layer in self.layers:

            x = layer(x)
            print(f'after {layer.__module__}: {x.shape}')

# x = df.iloc[:, 0:1].values.T
x = batch[1][0].numpy()
model = Classifier(jr.PRNGKey(42))
model.debug_call(x)

after equinox.nn._conv: (2, 113)
after equinox.nn._pool: (2, 112)
after jax._src.nn.functions: (2, 112)
after equinox.nn._conv: (4, 21)
after equinox.nn._pool: (4, 20)
after jax._src.nn.functions: (4, 20)
after equinox.nn._conv: (8, 2)
after equinox.nn._pool: (8, 1)
after jax._src.nn.functions: (8, 1)
after jax._src.numpy.lax_numpy: (8,)
after equinox.nn._linear: (3,)


In [132]:
# print(model)

In [135]:
y_pred = jax.vmap(model)(batch[1].numpy())
y_pred.shape

(32, 3)

In [109]:
def cross_entropy(
    y: Int[Array, " batch"], pred_y: Float[Array, "batch 3"]
) -> Float[Array, ""]:
    # y are the true targets, and should be integers 0-9.
    # pred_y are the log-softmax'd predictions.
    pred_y = jnp.take_along_axis(pred_y, jnp.expand_dims(y, 1), axis=1)
    return -jnp.mean(pred_y)

In [121]:
y = jnp.array([1, 2, 3])
y_pred = jnp.array([
    [0.1, 0.6, 0.1, 0.2],
    [0.1, 0.1, 0.6, 0.2],
    [0.1, 0.1, 0.1, 0.7]
])
print(y, '\n', y_pred)

[1 2 3] 
 [[0.1 0.6 0.1 0.2]
 [0.1 0.1 0.6 0.2]
 [0.1 0.1 0.1 0.7]]


In [122]:
cross_entropy(y, y_pred)

Array(-0.6333333, dtype=float32)

In [117]:
# optax.losses.softmax_cross_entropy_with_integer_labels(y_pred, y).mean()

Array(1.7928747, dtype=float32)

In [136]:
def loss(model: eqx.Module, x: Float[Array, 'batch length'], y: Float[Array, 'batch']):

    y_pred = jax.vmap(model)(x)
    
    return optax.losses.softmax_cross_entropy_with_integer_labels(y_pred, y).mean()

In [137]:

loss(model, batch[1].numpy(), batch[2].numpy())

Array(1.1721073, dtype=float32)

In [166]:
optim = optax.adamw(parameters['training']['learning_rate'])
opt_state = optim.init(eqx.filter(model, eqx.is_array))

In [141]:
@eqx.filter_jit
def take_step(
    model,
    opt_state: PyTree,
    x: Float[Array, 'batch 1 length'],
    y: Float[Array, 'batch']
) -> tuple[tuple[eqx.Module], PyTree, Float[Array, '']]:

    value, grads = eqx.filter_value_and_grad(loss)(model, x, y)

    # it returns transformed/processed gradients, and the optimizer state
    updates, opt_state = optim.update(grads, opt_state, model)

    model = eqx.apply_updates(model, updates)

    return model, opt_state, value

In [152]:
# model, opt_state, value = take_step(model, opt_state, batch[1].numpy(), batch[2].numpy())
# value

Array(0.76158667, dtype=float32)

In [167]:
n_epochs = 200

train_loss = [None] * n_epochs
valid_loss = [None] * n_epochs

In [168]:
for epoch in range(n_epochs):

    train_loss[epoch] = 0.

    for _, x, y in train_dl:

        key, subkey = jr.split(key)

        model, opt_state, loss_value = take_step(model, opt_state, x.numpy(), y.numpy())

        train_loss[epoch] += loss_value.item()

    # validation
    valid_loss[epoch] = 0.
    
    for _, x, y in validation_dl:
    
        loss_value = loss(model, x.numpy(), y.numpy())
        
        valid_loss[epoch] += valid_loss.item()

    valid_loss[epoch] /= len(validation_dl)

    print(epoch, train_loss[epoch] / len(train_dl), valid_loss[epoch])

0 0.38656039641963114 0.41880728647244714
1 0.3803364422917366 0.4145087538435396
2 0.3791669511795044 0.4044460246784497
3 0.3771887627575133 0.41343845110551447
4 0.37197291526529525 0.40502711945930414
5 0.3746709357202053 0.3991425831892849
6 0.3779253030485577 0.3998234130903683
7 0.3713621832099226 0.4062693202126343
8 0.3662505098846224 0.41132528394196943
9 0.3723984309534232 0.42008405317247444
10 0.37209910253683726 0.4033900753191087
11 0.362975840833452 0.4127994781706186
12 0.3683379554086261 0.41393773745646517
13 0.3695166864991188 0.4039837892735954
14 0.36087198904818957 0.41711982906655926
15 0.3677493368089199 0.41016503561914497
16 0.35672791906529 0.42945289690937616
17 0.3623781364162763 0.41393214009240664
18 0.3675827976895703 0.44836000814638305
19 0.35685801287492114 0.39605212666555845
20 0.36053584572341707 0.4142340850249856
21 0.3571645718316237 0.4089219139608662
22 0.35689616761273807 0.44662129694381647
23 0.3535249304274718 0.4076439912340282
24 0.3619